# Automated Meeting Minutes Generator

Converts a meeting transcript into a structured summary: 
1) key discussion points,
2) decisions made, and
3) action items (with owner and deadline when stated).

User has option to use LLM: OpenAI, Cohere, Gemini. It is a one-line change (see the config cell) via LangChain's `init_chat_model`.

## 1. Install

In [ ]:
%pip install -q langchain==1.3.6 langchain-google-genai==4.4.0 langchain-openai==1.2.2 langchain-cohere==0.6.0 pydantic python-dotenv

## 2. Imports and API keys

`.env` file next to the notebook (only the key for the provider you use is required):

```text
GOOGLE_API_KEY=...
OPENAI_API_KEY=...
COHERE_API_KEY=...
```

In [1]:
import os
import json
from typing import List, Optional

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

True

## 3. Config: switch provider here

Set `PROVIDER` to `"gemini"`, `"openai"`, or `"cohere"`. Edit `MODELS` if you want a different model for a provider.

In [2]:
PROVIDER = "gemini"          # Change provider here: "gemini" | "openai" | "cohere"

TEMPERATURE = 0.2
MAX_RETRIES = 3

MODELS = {
    "gemini": "gemini-2.5-flash",
    "openai": "gpt-4o-mini",
    "cohere": "command-a-03-2025",
}

# provider key -> (init_chat_model provider id, required env var)
PROVIDER_INFO = {
    "gemini": ("google_genai", "GOOGLE_API_KEY"),
    "openai": ("openai", "OPENAI_API_KEY"),
    "cohere": ("cohere", "COHERE_API_KEY"),
}

## 4. Build the model

In [3]:
def build_llm(provider: str):
    if provider not in PROVIDER_INFO:
        raise ValueError(f"Unknown provider '{provider}'. Choose from {list(PROVIDER_INFO)}")

    model_provider, env_var = PROVIDER_INFO[provider]

    if not os.getenv(env_var):
        raise ValueError(f"{env_var} not found. Add it to your .env file.")

    model_name = MODELS[provider]
    kwargs = {}

    # OpenAI reasoning models (gpt-5*, o*) reject a custom temperature
    is_openai_reasoning = provider == "openai" and model_name.startswith(("gpt-5", "o1", "o3", "o4"))
    if not is_openai_reasoning:
        kwargs["temperature"] = TEMPERATURE

    return init_chat_model(model=model_name, model_provider=model_provider, **kwargs)


llm = build_llm(PROVIDER)
print(f"Provider: {PROVIDER} | Model: {MODELS[PROVIDER]}")

Provider: gemini | Model: gemini-2.5-flash


## 5. Output schema

In [4]:
class ActionItem(BaseModel):
    task: str = Field(description="The action to be done")
    owner: Optional[str] = Field(default=None, description="Person or team responsible, if stated in the transcript")
    deadline: Optional[str] = Field(default=None, description="Due date or timeframe, if stated in the transcript")


class Decision(BaseModel):
    decision: str = Field(description="A decision that was made")
    context: Optional[str] = Field(default=None, description="Brief context or reasoning behind the decision, if given")


class MeetingMinutes(BaseModel):
    title: str = Field(description="A short descriptive title for the meeting")
    attendees: List[str] = Field(default_factory=list, description="Names mentioned as present, if identifiable from the transcript")
    summary: str = Field(description="A concise 3-5 sentence summary of the meeting")
    key_topics: List[str] = Field(description="Main topics discussed, as short bullet points")
    decisions: List[Decision] = Field(default_factory=list, description="Decisions made during the meeting")
    action_items: List[ActionItem] = Field(default_factory=list, description="Concrete follow-up tasks")
    open_questions: List[str] = Field(default_factory=list, description="Unresolved questions or items to follow up on")

## 6. Prompt and chain

In [5]:
system_prompt = """
    You are an assistant that writes accurate, concise meeting minutes from a transcript.
    
    Rules:
    - Use only information present in the transcript. Do not invent names, dates, or decisions.
    - If attendees are not identifiable, return an empty list rather than guessing.
    - Action items should be concrete and actionable. Include owner and deadline only if explicitly stated or clearly implied; otherwise leave them empty.
    - Keep the summary factual and neutral in tone.
    - List key topics as short phrases, not full sentences.
    - Separate decisions (things agreed/resolved) from action items (things to be done).
    - If nothing fits a field (e.g. no open questions), return an empty list for it.
"""

human_prompt =  """
    Meeting transcript:
    {transcript}
    
    Generate the meeting minutes.
"""

prompt = ChatPromptTemplate.from_messages([
    ( "system",system_prompt,),
    ( "human",human_prompt,),
])

structured_llm = llm.with_structured_output(MeetingMinutes)
chain = prompt | structured_llm

## 7. Generation function

Retries on malformed output. Long transcripts are chunked by character count and summarized in passes so very long meetings don't overflow the model's context window.

In [6]:
MAX_CHARS_PER_CHUNK = 12000


def chunk_transcript(transcript: str, max_chars: int = MAX_CHARS_PER_CHUNK) -> List[str]:
    if len(transcript) <= max_chars:
        return [transcript]

    lines = transcript.splitlines(keepends=True)
    chunks, current = [], ""

    for line in lines:
        if len(current) + len(line) > max_chars and current:
            chunks.append(current)
            current = ""
        current += line

    if current:
        chunks.append(current)

    return chunks


def _generate_once(transcript: str) -> MeetingMinutes:
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            result = chain.invoke({"transcript": transcript})
            if isinstance(result, dict):
                result = MeetingMinutes.model_validate(result)
            return result
        except Exception as e:
            last_error = e
            print(f"Attempt {attempt} failed: {e}")

    raise RuntimeError(f"Minutes generation failed after {MAX_RETRIES} attempts: {last_error}")


def generate_minutes(transcript: str) -> MeetingMinutes:
    if not transcript.strip():
        raise ValueError("Transcript is empty.")

    chunks = chunk_transcript(transcript)

    if len(chunks) == 1:
        return _generate_once(chunks[0])

    # Summarize each chunk, then merge the partial minutes into one transcript-like
    # text and run generation again for a coherent final result.
    print(f"Transcript split into {len(chunks)} chunks; summarizing each...")
    partials = [_generate_once(c) for c in chunks]

    merged_text = "\n\n".join(
        f"Partial summary {i+1}:\n{p.model_dump_json(indent=2)}"
        for i, p in enumerate(partials)
    )

    return _generate_once(merged_text)

## 8. Display and export

In [7]:
def display_minutes(minutes: MeetingMinutes) -> None:
    print(minutes.title)
    print("=" * 60)

    if minutes.attendees:
        print("Attendees:", ", ".join(minutes.attendees))

    print("\nSUMMARY")
    print("-" * 60)
    print(minutes.summary)

    print("\nKEY TOPICS")
    print("-" * 60)
    for topic in minutes.key_topics:
        print(f"- {topic}")

    print("\nDECISIONS")
    print("-" * 60)
    if minutes.decisions:
        for d in minutes.decisions:
            line = f"- {d.decision}"
            if d.context:
                line += f" ({d.context})"
            print(line)
    else:
        print("None recorded.")

    print("\nACTION ITEMS")
    print("-" * 60)
    if minutes.action_items:
        for a in minutes.action_items:
            owner = a.owner or "Unassigned"
            deadline = f" — due {a.deadline}" if a.deadline else ""
            print(f"- [{owner}] {a.task}{deadline}")
    else:
        print("None recorded.")

    print("\nOPEN QUESTIONS")
    print("-" * 60)
    if minutes.open_questions:
        for q in minutes.open_questions:
            print(f"- {q}")
    else:
        print("None recorded.")


def save_minutes(minutes: MeetingMinutes, path: str = "meeting_minutes.json") -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(minutes.model_dump(), f, indent=2, ensure_ascii=False)
    print(f"Saved to {path}")


def minutes_to_markdown(minutes: MeetingMinutes) -> str:
    lines = [f"# {minutes.title}", ""]

    if minutes.attendees:
        lines += [f"**Attendees:** {', '.join(minutes.attendees)}", ""]

    lines += ["## Summary", minutes.summary, ""]

    lines += ["## Key Topics"]
    lines += [f"- {t}" for t in minutes.key_topics] or ["- None recorded."]
    lines.append("")

    lines += ["## Decisions"]
    lines += ([f"- {d.decision}" + (f" ({d.context})" if d.context else "") for d in minutes.decisions]
              or ["- None recorded."])
    lines.append("")

    lines += ["## Action Items"]
    lines += ([f"- [ ] **{a.owner or 'Unassigned'}**: {a.task}" + (f" (due {a.deadline})" if a.deadline else "")
               for a in minutes.action_items]
              or ["- None recorded."])
    lines.append("")

    lines += ["## Open Questions"]
    lines += [f"- {q}" for q in minutes.open_questions] or ["- None recorded."]

    return "\n".join(lines)


def save_minutes_markdown(minutes: MeetingMinutes, path: str = "meeting_minutes.md") -> None:
    with open(path, "w", encoding="utf-8") as f:
        f.write(minutes_to_markdown(minutes))
    print(f"Saved to {path}")

## 9. Add your transcript

Paste a transcript below, or load it from a `.txt` file (see the next cell).

In [8]:
TRANSCRIPT = """
Alex: Thanks everyone for joining. Let's start with the Q3 launch timeline.
Priya: The backend is on track, but we're blocked on the payments API review from legal.
Alex: Okay, let's flag that. Priya, can you follow up with legal by Thursday?
Priya: Sure, I'll email them today.
Sam: On the design side, we finalized the onboarding flow. I'll share the Figma link after this call.
Alex: Great. Let's decide: are we launching in three regions or just one for the beta?
Sam: I'd push for just one region first, lower risk.
Priya: Agreed, let's start with just the US.
Alex: Okay, decision made — beta launches in the US only.
Sam: One open question: do we have budget for the paid ad campaign at launch?
Alex: Not sure yet, I'll check with finance and get back to everyone next week.
Alex: Anything else? No? Let's wrap up. Priya follows up with legal by Thursday, Sam shares the Figma link today, and I'll check on ad budget by next week.
"""

In [ ]:
# Optional: load transcript from a file instead
# with open("transcript.txt", encoding="utf-8") as f:
#     TRANSCRIPT = f.read()

## 10. Generate

In [9]:
minutes = generate_minutes(TRANSCRIPT)
display_minutes(minutes)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Q3 Launch Timeline and Beta Planning
Attendees: Alex, Priya, Sam

SUMMARY
------------------------------------------------------------
The meeting focused on the Q3 launch timeline. The backend is on track, but the payments API review by legal is a blocker. The onboarding flow design has been finalized. A key decision was made to launch the beta in the US only to mitigate risk. The budget for a paid ad campaign at launch remains an open question.

KEY TOPICS
------------------------------------------------------------
- Q3 launch timeline
- Payments API review
- Onboarding flow design
- Beta launch regions
- Paid ad campaign budget

DECISIONS
------------------------------------------------------------
- Beta will launch in the US only. (To lower risk, it was agreed to start with one region for the beta.)

ACTION ITEMS
------------------------------------------------------------
- [Priya] Follow up with legal regarding the payments API review. — due Thursday
- [Sam] Share the Figma lin

In [ ]:
# Optional: save as JSON and/or Markdown
# save_minutes(minutes, "meeting_minutes.json")
# save_minutes_markdown(minutes, "meeting_minutes.md")

## Switching providers

Change one line in the config cell and re-run from there:

```python
PROVIDER = "openai"   # or "cohere" or "gemini"
```

To add a new provider, add one entry each to `MODELS` and `PROVIDER_INFO` (any provider supported by `init_chat_model`).

# STOP

# Further Enhancement

**Easy**
- Multiple export formats — Word (`.docx`) or PDF output for actually sending minutes to attendees, not just JSON/Markdown.
- Priority/urgency tagging — add a `priority: Literal["low","medium","high"]` field to `ActionItem`.
- Timestamps in output — if the transcript includes timestamps, carry them through so each topic/decision links back to "discussed at 14:32."
- Email-ready summary — a formatter that turns `MeetingMinutes` into a ready-to-send email body (subject line + body), since that's the actual end use case for most people.
- Multiple meeting types/templates — a standup transcript needs different fields (blockers, yesterday/today) than a client call (next steps, risks) — make the schema swappable per meeting type.

**Medium**
- Speaker-aware parsing — currently it treats the whole transcript as one blob; explicitly parsing `"Name: text"` lines lets the model reliably attribute decisions/action items to who actually said them, instead of guessing.
- Action item deduplication across meetings — if the same action item ("follow up with legal") appears in consecutive weekly meetings, detect it's a carryover rather than a new item.
- Sentiment/tone flagging — flag if a discussion got contentious or if a decision was made without consensus (useful for team leads reviewing minutes later).
- Calendar/task integration — push generated action items directly to Google Calendar, Asana, or Trello via their APIs instead of just printing them (this is where the LangChain MCP tool pattern would fit well).
- Follow-up reminder generator — auto-draft a "reminder" message for each action item nearing its deadline.

**Harder**
- Diarization from raw audio — right now it assumes a clean text transcript; students could add a speech-to-text + speaker-diarization step (e.g. Whisper + pyannote) so the input is an actual audio recording of the meeting.
- Chunking, done properly — the current chunking is naive (split by character count, then merge partial summaries). A better version would chunk by topic boundaries (detect topic shifts) rather than raw length, so a decision made right at a chunk boundary doesn't get split awkwardly.
- Multi-meeting trend analysis — across a series of meetings for the same project, track which action items keep slipping, which decisions get revisited, and surface that as a report.
- Accuracy validation — extractive grounding, similar to the code-explainer suggestion: have the model cite the exact line(s) supporting each decision/action item, so a human can spot-check instead of trusting it blindly.
- Real-time mode — instead of a full transcript after the fact, feed it a live streaming transcript and have it incrementally update minutes as the meeting happens.
